# 微调、分布式训练与推理优化
## Fine-tuning, Distributed Training and Inference Optimization


<img src="../images/logo.png" width=150>

本文档补充项目中未覆盖的重要主题：微调方法（SFT/RLHF/DPO/LoRA）、分布式训练（ZeRO/FSDP）和推理优化（Paged Attention/Speculative Decoding）。

This document covers important topics not detailed in other notebooks: fine-tuning methods (SFT/RLHF/DPO/LoRA), distributed training (ZeRO/FSDP), and inference optimization (Paged Attention/Speculative Decoding).


# 1. 微调方法概述
## 1. Fine-tuning Methods Overview


| 方法 | 全称 | 核心思想 | 适用场景 |
|------|------|----------|----------|
| **SFT** | Supervised Fine-Tuning | 人工标注的问答对直接监督学习 | 通用对话、问答 |
| **RLHF** | Reinforcement Learning from Human Feedback | 人类偏好训练reward模型，再用PPO优化 | 对齐人类价值观 |
| **DPO** | Direct Preference Optimization | 直接用偏好数据优化策略绕过reward模型 | 简化RLHF流程 |
| **LoRA** | Low-Rank Adaptation | 冻结原模型，注入低秩矩阵 | 参数高效微调 |


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np


# 2. LoRA（低秩适配）实现
## 2. LoRA (Low-Rank Adaptation) Implementation


In [ ]:
class LoRALinear(nn.Module):
    """
    LoRA实现：冻结原权重，注入低秩矩阵
    LoRA implementation: freeze original weights, inject low-rank matrices
    
    原权重 W ∈ R^{d×k}
    LoRA: W + BA，其中 B ∈ R^{d×r}, A ∈ R^{r×k}, r << min(d,k)
    """
    def __init__(self, in_features, out_features, rank=4, alpha=1.0):
        super().__init__()
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank
        
        # 原始权重冻结 / Original weights frozen
        self.weight = None
        self.bias = None
        
        # LoRA低秩矩阵 / LoRA low-rank matrices
        self.lora_A = nn.Parameter(torch.randn(in_features, rank))
        self.lora_B = nn.Parameter(torch.zeros(rank, out_features))
        
        self._init_lora()
    
    def _init_lora(self):
        nn.init.normal_(self.lora_A, std=0.02)
        nn.init.zeros_(self.lora_B)
    
    def forward(self, x):
        original = F.linear(x, self.weight, self.bias)
        lora = x @ self.lora_A @ self.lora_B
        return original + lora * self.scaling
    
    def merge_weights(self):
        if self.weight is not None:
            self.weight.data += (self.lora_A @ self.lora_B).t() * self.scaling
            self.lora_A = None
            self.lora_B = None


In [ ]:
# 测试LoRA / Test LoRA
layer = LoRALinear(in_features=512, out_features=512, rank=4, alpha=8)
original_weights = nn.Parameter(torch.randn(512, 512))
layer.weight = original_weights
x = torch.randn(2, 10, 512)
output = layer(x)
print(f"LoRA Layer input: {x.shape}")
print(f"LoRA Layer output: {output.shape}")
print(f"Trainable params: {sum(p.numel() for p in layer.parameters() if p.requires_grad):,}")
print(f"Frozen params: {layer.weight.numel():,}")


# 3. LoRA可视化
## 3. LoRA Visualization


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax1 = axes[0]
methods = ["Full Fine-tune", "LoRA r=4", "LoRA r=8", "LoRA r=16", "LoRA r=64"]
params_percent = [100, 0.11, 0.22, 0.45, 1.8]

bars = ax1.barh(methods, params_percent, color=["#e74c3c", "#3498db", "#2ecc71", "#9b59b6", "#f39c12"])
ax1.set_xlabel("Trainable Parameters (% of model)")
ax1.set_title("LoRA Parameter Efficiency\n(Lower is better)")

for bar, pct in zip(bars, params_percent):
    ax1.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
             f"{pct:.2f}%", va="center", fontsize=9)

ax2 = axes[1]
ranks = [2, 4, 8, 16, 32, 64, 128]
embed_dim = 4096
lora_params = [2 * embed_dim * r + 2 * r * embed_dim for r in ranks]
original_params = embed_dim * embed_dim

ax2.plot(ranks, [p/1e6 for p in lora_params], "b-o", linewidth=2, markersize=8, label="LoRA params (M)")
ax2.axhline(y=original_params/1e6, color="r", linestyle="--", label=f"Full ({original_params/1e6:.0f}M)")
ax2.set_xlabel("LoRA Rank")
ax2.set_ylabel("Parameters (M)")
ax2.set_title("LoRA Rank vs Parameters\n(Rank << dim)")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../images/lora_visualization.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"LoRA r=4 vs Full: {(lora_params[1]/original_params)*100:.3f}% of parameters")


# 4. 分布式训练策略
## 4. Distributed Training Strategies


## 4.1 ZeRO（Zero Redundancy Optimizer）

ZeRO通过分片优化器状态、梯度、参数来减少显存冗余：

| Stage | 优化内容 | 显存节省 |
|-------|----------|----------|
| ZeRO-1 | 分片优化器状态 | ~4x |
| ZeRO-2 | 分片优化器状态 + 梯度 | ~8x |
| ZeRO-3 | 分片所有状态（参数+梯度+优化器） | ~N倍，N=GPU数 |


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

model_size_gb = 7
gpu_memory_fp32 = model_size_gb * 4

ax1 = axes[0]
labels1 = ["Model", "Optimizer", "Gradients", "Activations"]
sizes1 = [28, 28, 28, 16]
colors1 = ["#3498db", "#2ecc71", "#e74c3c", "#f39c12"]
ax1.pie(sizes1, labels=labels1, colors=colors1, autopct="%1.1f%%")
ax1.set_title("No ZeRO\n(Fully replicated)")

ax2 = axes[1]
sizes2 = [28, 4, 4, 16]
ax2.pie(sizes2, labels=labels1, colors=colors1, autopct="%1.1f%%")
ax2.set_title("ZeRO-2\n(Sharded optimizer + gradients)")

ax3 = axes[2]
sizes3 = [4, 4, 4, 16]
ax3.pie(sizes3, labels=labels1, colors=colors1, autopct="%1.1f%%")
ax3.set_title("ZeRO-3\n(Fully sharded)")

plt.tight_layout()
plt.savefig("../images/zero_stages.png", dpi=150, bbox_inches="tight")
plt.show()

print("ZeRO显存优化效果：")
print(f"  7B模型 FP32: {model_size_gb * 4}GB")
print(f"  ZeRO-1: {model_size_gb * 4 / 4:.1f}GB per GPU")
print(f"  ZeRO-2: {model_size_gb * 4 / 8:.1f}GB per GPU")


## 4.2 Pipeline Parallelism（流水线并行）


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

num_stages = 4
num_microbatches = 8

for stage in range(num_stages):
    for micro in range(num_microbatches):
        start_time = stage + micro
        if start_time < num_microbatches:
            ax.barh(stage, 0.8, left=start_time, color=plt.cm.viridis(micro/num_microbatches))

ax.set_xlabel("Time (microbatches)")
ax.set_ylabel("Stage")
ax.set_yticks(range(num_stages))
ax.set_yticklabels([f"Stage {i+1}" for i in range(num_stages)])
ax.set_title("Pipeline Parallelism (4 stages, 8 microbatches)\n(Color = different microbatches)")

plt.tight_layout()
plt.savefig("../images/pipeline_parallelism.png", dpi=150, bbox_inches="tight")
plt.show()

print("Pipeline parallelism特点:")
print("  - 将模型层划分到不同GPU")
print("  - 多个microbatch流水处理提高GPU利用率")
print("  - 存在流水线气泡开销")


# 5. 推理优化
## 5. Inference Optimization


## 5.1 Paged Attention（分页注意力）


In [ ]:
class PagedKVCache:
    """
    Paged KV Cache实现 - 显存管理优化
    Paged KV Cache implementation - memory management optimization
    """
    def __init__(self, num_heads, head_dim, block_size=16, max_blocks=256):
        self.block_size = block_size
        self.num_heads = num_heads
        self.head_dim = head_dim
        self.blocks = {}
        self.free_blocks = set(range(max_blocks))
        self.max_blocks = max_blocks
    
    def allocate(self, seq_len):
        num_blocks = (seq_len + self.block_size - 1) // self.block_size
        block_ids = []
        for i in range(num_blocks):
            if not self.free_blocks:
                raise RuntimeError("Out of KV cache blocks")
            block_id = self.free_blocks.pop()
            self.blocks[block_id] = torch.zeros(
                self.block_size, self.num_heads, self.head_dim
            )
            block_ids.append(block_id)
        return block_ids
    
    def write(self, block_ids, start_pos, values):
        for i, v in enumerate(values):
            block_id = block_ids[start_pos // self.block_size]
            offset = start_pos % self.block_size
            self.blocks[block_id][offset] = v
    
    def read(self, block_ids, start_pos, seq_len):
        result = []
        for i in range(seq_len):
            block_id = block_ids[start_pos // self.block_size]
            offset = start_pos % self.block_size
            result.append(self.blocks[block_id][offset])
        return torch.stack(result)
    
    def free(self, block_ids):
        for block_id in block_ids:
            if block_id in self.blocks:
                del self.blocks[block_id]
                self.free_blocks.add(block_id)


In [ ]:
paged_cache = PagedKVCache(num_heads=8, head_dim=64, block_size=16)
seq1_blocks = paged_cache.allocate(32)
print(f"Sequence 1 allocated blocks: {seq1_blocks}")
seq2_blocks = paged_cache.allocate(64)
print(f"Sequence 2 allocated blocks: {seq2_blocks}")
print(f"\nFree blocks remaining: {len(paged_cache.free_blocks)}")
print(f"Active blocks: {list(paged_cache.blocks.keys())}")


## 5.2 Speculative Decoding（推测解码）


In [ ]:
class SpeculativeDecoder:
    """
    推测解码：使用小模型推测多个token，大模型验证
    Speculative decoding: use small model to draft tokens, large model to verify
    """
    def __init__(self, draft_model, target_model, k=4, beta=0.5):
        self.draft = draft_model
        self.target = target_model
        self.k = k
        self.beta = beta
    
    def decode_step(self, x):
        draft_tokens = []
        draft_probs = []
        current = x
        
        for _ in range(self.k):
            draft_out = self.draft(current)
            draft_token = draft_out.argmax(dim=-1)[:, -1]
            draft_prob = F.softmax(draft_out[:, -1], dim=-1)
            draft_tokens.append(draft_token)
            draft_probs.append(draft_prob)
            current = torch.cat([current, draft_token.unsqueeze(-1)], dim=-1)
        
        target_out = self.target(current)
        target_probs = F.softmax(target_out[:, -self.k:], dim=-1)
        
        accepted = 0
        for i in range(self.k):
            p_target = target_probs[:, i, draft_tokens[i]]
            p_draft = draft_probs[i][:, draft_tokens[i]]
            acceptance_ratio = p_target / (p_draft + 1e-10)
            if acceptance_ratio > self.beta:
                accepted += 1
            else:
                break
        
        final_len = min(x.shape[1] + accepted + 1, current.shape[1])
        output = current[:, :final_len]
        
        return output, accepted


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax1 = axes[0]
beta_values = np.linspace(0.1, 0.9, 20)
acceptance_rates = [1 / (1 + 2 * (1 - b)) for b in beta_values]

ax1.plot(beta_values, acceptance_rates, "b-o", linewidth=2)
ax1.set_xlabel("Beta (acceptance threshold)")
ax1.set_ylabel("Token Acceptance Rate")
ax1.set_title("Speculative Decoding: Beta vs Acceptance")
ax1.grid(True, alpha=0.3)

ax2 = axes[1]
draft_speeds = [2, 4, 8, 16]
speedups = [s / (1 + (s-1)*(1-0.8)) for s in draft_speeds]

ax2.bar(range(len(draft_speeds)), speedups, color=["#3498db", "#2ecc71", "#e74c3c", "#9b59b6"])
ax2.set_xticks(range(len(draft_speeds)))
ax2.set_xticklabels([f"{s}x draft" for s in draft_speeds])
ax2.set_ylabel("Speedup Factor")
ax2.set_title("Speculative Decoding Speedup\n(80% acceptance rate)")
ax2.axhline(y=1, color="gray", linestyle="--")

for i, su in enumerate(speedups):
    ax2.text(i, su + 0.1, f"{su:.1f}x", ha="center", fontsize=10)

plt.tight_layout()
plt.savefig("../images/speculative_decoding.png", dpi=150, bbox_inches="tight")
plt.show()

print("Speculative Decoding效果:")
print("  - 理想情况：2-4x加速")
print("  - 关键指标：推测token接受率")


# 6. 完整技术对比
## 6. Complete Technology Comparison


In [ ]:
print("="*60)
print("LLM训练与推理优化技术对比")
print("="*60)

techs = [
    ("SFT", "直接监督学习", "简单有效", "标注成本高"),
    ("RLHF", "人类反馈强化学习", "对齐人类价值观", "训练不稳定"),
    ("DPO", "直接偏好优化", "绕过reward模型", "需要高质量偏好数据"),
    ("LoRA", "低秩适配", "参数高效(0.1%-2%)", "表达能力有限"),
    ("ZeRO-1", "优化器分片", "4x显存减少", "通信开销"),
    ("ZeRO-2", "梯度分片", "8x显存减少", "通信开销增加"),
    ("ZeRO-3", "全量分片", "线性扩展", "通信开销高"),
    ("PagedAttention", "KV cache分页", "显存利用率提升", "实现复杂"),
    ("SpeculativeDecoding", "推测解码", "2-4x生成加速", "需小模型配合"),
]

print(f"\n{'技术':<20} {'核心思想':<15} {'优势':<20} {'局限'}")
print("-"*70)
for name, idea, pro, con in techs:
    print(f"{name:<20} {idea:<15} {pro:<20} {con}")


# 总结
## Summary


| 类别 | 技术 | 核心价值 |
|------|------|----------|
| 参数高效微调 | LoRA, QLoRA | 降低微调成本 |
| 分布式训练 | ZeRO, FSDP, Pipeline | 支持超大规模模型 |
| 推理优化 | Paged Attention, Speculative Decoding | 降低延迟提升吞吐 |


# 已实现 / Implemented

本notebook已完整实现以下内容：

1. **LoRA** - 低秩适配层实现
2. **ZeRO分片策略** - 可视化分片效果
3. **Pipeline Parallelism** - 流水线并行概念
4. **Paged KV Cache** - 分页缓存管理
5. **Speculative Decoding** - 推测解码框架

## 扩展阅读 / Further Reading

| 主题 | 说明 | 推荐资源 |
|------|------|----------|
| **LLaMA Factory** | 微调工具 | [LLaMA Factory](https://github.com/hiyouga/LLaMA-Factory) |
| **DeepSpeed** | 分布式训练 | [DeepSpeed](https://www.deepspeed.ai/) |
| **vLLM** | 推理优化 | [vLLM](https://vllm.ai/) |
| **trl** | RLHF/DPO实现 | [TRL Library](https://github.com/huggingface/trl) |
| **Axolotl** | 微调框架 | [Axolotl](https://github.com/OpenAccess-AI-Collective/axolotl) |
